In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/module1_spot_v0.parquet", engine="fastparquet")

df["ts"] = pd.to_datetime(df["ts"], utc=True)
df = df.sort_values("ts").reset_index(drop=True)

# --- Returns ---
df["log_return"] = np.log(df["close"] / df["close"].shift(1))

# --- Volatility (multi-scale) ---
df["vol_5"]  = df["log_return"].rolling(5).std()
df["vol_20"] = df["log_return"].rolling(20).std()
df["vol_60"] = df["log_return"].rolling(60).std()

# --- Trend (multi-horizon returns) ---
df["ret_3"]  = df["close"].pct_change(3)
df["ret_7"]  = df["close"].pct_change(7)
df["ret_14"] = df["close"].pct_change(14)

# --- Moving averages (trend strength proxy) ---
df["ma_10"] = df["close"].rolling(10).mean()
df["ma_30"] = df["close"].rolling(30).mean()

df["trend_strength"] = (df["ma_10"] - df["ma_30"]) / df["ma_30"]

# --- Distribution features ---
df["skew_20"] = df["log_return"].rolling(20).skew()
df["kurt_20"] = df["log_return"].rolling(20).kurt()

# --- Range / volatility proxy ---
df["hl_range"] = (df["high"] - df["low"]) / df["close"]
df["range_5"]  = df["hl_range"].rolling(5).mean()

# --- Volume features ---
df["vol_zscore"] = (df["volume"] - df["volume"].rolling(20).mean()) / df["volume"].rolling(20).std()

# --- Cleanup ---
df = df.dropna().reset_index(drop=True)

In [2]:
# df

In [3]:
# --- Parameters ---
HOLDING = 5
TP_MULT = 2.0
SL_MULT = 1.0

df = df.copy()

# --- Volatility (use vol_20 as sigma) ---
df["sigma"] = df["log_return"].rolling(20).std()

# --- Pre-allocate ---
labels = []
exit_times = []
returns = []

close = df["close"].values
ts = df["ts"].values
sigma = df["sigma"].values

n = len(df)

for i in range(n - HOLDING):
    
    entry_px = close[i]
    entry_sigma = sigma[i]
    
    if np.isnan(entry_sigma):
        labels.append(np.nan)
        exit_times.append(pd.NaT)
        returns.append(np.nan)
        continue
    
    tp = entry_px * (1 + TP_MULT * entry_sigma)
    sl = entry_px * (1 - SL_MULT * entry_sigma)
    
    labels = [np.nan] * n
exit_times = [pd.NaT] * n
returns = [np.nan] * n

for i in range(n - HOLDING):
    
    entry_px = close[i]
    entry_sigma = sigma[i]
    
    if np.isnan(entry_sigma):
        continue
    
    tp = entry_px * (1 + TP_MULT * entry_sigma)
    sl = entry_px * (1 - SL_MULT * entry_sigma)
    
    label = 0
    exit_time = ts[i + HOLDING]
    ret = 0
    
    for j in range(1, HOLDING + 1):
        px = close[i + j]
        
        if px >= tp:
            label = 1
            exit_time = ts[i + j]
            ret = (px - entry_px) / entry_px
            break
        
        elif px <= sl:
            label = 0
            exit_time = ts[i + j]
            ret = (px - entry_px) / entry_px
            break
        
        if j == HOLDING:
            px = close[i + j]
            ret = (px - entry_px) / entry_px
    
    labels[i] = label
    exit_times[i] = exit_time
    returns[i] = ret

# --- Assign ---
df["tb_label"] = labels
df["tb_exit_time"] = exit_times
df["tb_return"] = returns

# --- Cleanup ---
df = df.dropna(subset=["tb_label"]).reset_index(drop=True)

In [4]:
df["tb_label"].value_counts(normalize=True)

tb_label
0.0    0.759287
1.0    0.240713
Name: proportion, dtype: float64

In [5]:
# df.columns

In [6]:
features = [
    "vol_5", "vol_20", "vol_60",
    "ret_3", "ret_7", "ret_14",
    "trend_strength",
    "skew_20", "kurt_20",
    "range_5", "vol_zscore"
]

sep = df.groupby("tb_label")[features].mean().T
sep["diff"] = sep[1] - sep[0]

sep.sort_values("diff", ascending=False)

tb_label,0.0,1.0,diff
skew_20,-0.012525,0.174452,0.186977
vol_zscore,-0.021758,0.133153,0.154911
ret_14,0.016466,0.053997,0.037531
ret_7,0.007603,0.026987,0.019383
trend_strength,0.007052,0.025749,0.018697
ret_3,0.003351,0.010422,0.007071
range_5,0.052720,0.051382,-0.001338
vol_60,0.035227,0.033607,-0.001620
vol_5,0.031685,0.028525,-0.003160
vol_20,0.034348,0.030636,-0.003712


In [7]:
df["high_skew"] = (df["skew_20"] > df["skew_20"].median()).astype(int)
df["high_vol_activity"] = (df["vol_zscore"] > 0).astype(int)
df["trend_up"] = (df["trend_strength"] > 0).astype(int)

In [8]:
# --- Rolling drawdown ---
df["roll_max_20"] = df["close"].rolling(20).max()
df["drawdown_20"] = (df["close"] - df["roll_max_20"]) / df["roll_max_20"]

# --- Volatility spike ---
df["vol_spike"] = df["vol_5"] / df["vol_20"]

# --- Smooth extreme noise ---
df["vol_spike"] = df["vol_spike"].clip(upper=5)
df["ma_50"] = df["close"].rolling(50).mean()
df["ma_200"] = df["close"].rolling(200).mean()

df["macro_trend"] = (df["ma_50"] > df["ma_200"]).astype(int)
# --- Risk regime (crash / unstable) ---
df["risk_regime"] = (
    (df["drawdown_20"] < -0.10) |
    (df["vol_spike"] > 1.5) |
    (df["macro_trend"] == 0)   # <-- NEW
).astype(int)

In [9]:
df["favorable_regime"] = (
    (df["high_skew"] == 1) &
    (df["trend_up"] == 1)
).astype(int)

In [10]:
df.groupby("favorable_regime")["tb_label"].mean()

favorable_regime
0    0.216071
1    0.283925
Name: tb_label, dtype: float64

In [11]:
df["strong_regime"] = (
    (df["skew_20"] > df["skew_20"].quantile(0.7)) &
    (df["trend_strength"] > df["trend_strength"].quantile(0.7)) &
    (df["vol_zscore"] > 0)
).astype(int)

In [12]:
df.groupby("strong_regime")["tb_label"].mean()

strong_regime
0    0.232397
1    0.353591
Name: tb_label, dtype: float64

In [13]:
df.groupby("risk_regime")["tb_label"].mean()

risk_regime
0    0.28149
1    0.21913
Name: tb_label, dtype: float64

In [14]:
import pandas as pd
import numpy as np

# --- Filter strong regime ---
train_df = df[df["strong_regime"] == 1].copy()

# --- Features ---
features = [
    "skew_20",
    "vol_zscore",
    "ret_14",
    "ret_7",
    "trend_strength"
]

train_df = train_df.dropna(subset=features + ["tb_label"]).reset_index(drop=True)

In [15]:
# Example: 80% train, 20% test (time-ordered)
split_idx = int(len(train_df) * 0.8)

train = train_df.iloc[:split_idx]
test  = train_df.iloc[split_idx:]

X_train = train[features]
y_train = train["tb_label"]

X_test = test[features]
y_test = test["tb_label"]

In [16]:
# !pip install xgboost
# !pip install scikit-learn==1.4.2

In [17]:
from xgboost import XGBClassifier
import xgboost as xgb
model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
test = test.copy()
test["prob"] = model.predict_proba(X_test)[:, 1]

In [19]:
# THRESHOLD = 0.6  # initial guess

# # trades = test[test["prob"] >= THRESHOLD].copy()

In [20]:
# trades = test[
#     (test["strong_regime"] == 1) 
#     # (test["prob"] >= THRESHOLD)
# ]

In [21]:
# subset = test[
#     (test["strong_regime"] == 1) &
#     (test["risk_regime"] == 0) &   # <-- NEW
#     (test["prob"] >= THRESHOLD)
# ].copy()

In [22]:
# trades = trades.sort_values("tb_exit_time")

# trades["cum_pnl"] = trades["tb_return"].cumsum()
# trades["peak"] = trades["cum_pnl"].cummax()
# trades["drawdown"] = trades["cum_pnl"] - trades["peak"]

# mean_ret = trades["tb_return"].mean()
# sharpe = (
#     mean_ret / trades["tb_return"].std() * np.sqrt(252)
#     if trades["tb_return"].std() > 0 else np.nan
# )
# max_dd = trades["drawdown"].min()

# mean_ret, sharpe, max_dd, len(trades)

In [23]:
# results = []

# for t in np.arange(0.5, 0.8, 0.02):
    
#     subset = test[
#         (test["strong_regime"] == 1) &
#         (test["prob"] >= t)
#     ].copy()
    
#     subset = subset.dropna(subset=["tb_return"])
    
#     if len(subset) < 10:
#         continue
    
#     ret = subset["tb_return"]
    
#     sharpe = np.nan
#     if ret.std() > 0:
#         sharpe = ret.mean() / ret.std() * np.sqrt(252)
    
#     results.append({
#         "threshold": t,
#         "trades": len(subset),
#         "mean_ret": ret.mean(),
#         "sharpe": sharpe
#     })

# res_df = pd.DataFrame(results)

# res_df

In [24]:
results = []

for t in np.arange(0.2, 0.5, 0.02):
    
    subset = test[
        (test["strong_regime"] == 1) &
        (test["risk_regime"] == 0) &
        (test["prob"] >= t)   # <-- FIXED
    ].copy()
    
    subset = subset.dropna(subset=["tb_return"])
    
    if len(subset) < 10:
        continue
    
    ret = subset["tb_return"]
    
    sharpe = np.nan
    if ret.std() > 0:
        sharpe = ret.mean() / ret.std() * np.sqrt(252)
    
    results.append({
        "threshold": t,
        "trades": len(subset),
        "mean_ret": ret.mean(),
        "sharpe": sharpe
    })

pd.DataFrame(results).sort_values("sharpe", ascending=False)

,threshold,trades,mean_ret,sharpe
10,0.40,11,0.046420,12.318386
11,0.42,10,0.041750,10.881659
9,0.38,12,0.037031,8.952821
7,0.34,14,0.028839,7.072098
6,0.32,14,0.028839,7.072098
8,0.36,14,0.028839,7.072098
1,0.22,18,0.025788,7.041716
2,0.24,17,0.024540,6.527930
0,0.20,19,0.023106,6.357732
3,0.26,15,0.025495,6.352698


In [25]:
print("Total test:", len(test))
print("Strong regime:", (test["strong_regime"] == 1).sum())

for t in [0.2, 0.3, 0.4]:
    count = len(test[
        (test["strong_regime"] == 1) &
        (test["prob"] >= t)
    ])
    print(f"Threshold {t}: {count} trades")

Total test: 37
Strong regime: 37
Threshold 0.2: 28 trades
Threshold 0.3: 22 trades
Threshold 0.4: 17 trades


In [26]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

In [27]:
FEATURES = [
    "skew_20",
    "vol_zscore",
    "ret_14",
    "ret_7",
    "trend_strength",
    "strong_regime"
]

THRESHOLD = 0.28   # from your sweep sweet spot
MIN_TRADES = 1

In [28]:
df = df.sort_values("ts").reset_index(drop=True)

df["year"] = df["ts"].dt.year

df = df.dropna(subset=FEATURES + ["tb_label", "tb_return"]).reset_index(drop=True)

years = sorted(df["year"].unique())

In [29]:
results = []
all_rows = []

for i in range(3, len(years)):
    
    train_years = years[:i]
    test_year = years[i]
    
    train = df[df["year"].isin(train_years)]
    test  = df[df["year"] == test_year].copy()
    
    if len(test) == 0:
        continue
    
    X_train = train[FEATURES]
    y_train = train["tb_label"]
    
    X_test = test[FEATURES]
    
    model = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    )
    
    model.fit(X_train, y_train)
    
    test["prob"] = model.predict_proba(X_test)[:, 1]
    
    # --- FINAL DECISION ---
    thr = test["prob"].quantile(0.7)

    subset = test[
        (test["strong_regime"] == 1) &
        (test["prob"] >= thr)
    ].copy()

    subset = subset.dropna(subset=["tb_return"])

    if len(subset) < MIN_TRADES:
        continue
    
    # --- Metrics ---
    subset = subset.sort_values("tb_exit_time")
    
    subset["cum"] = subset["tb_return"].cumsum()
    subset["peak"] = subset["cum"].cummax()
    subset["dd"] = subset["cum"] - subset["peak"]
    
    mean_ret = subset["tb_return"].mean()
    
    sharpe = np.nan
    if subset["tb_return"].std() > 0:
        sharpe = mean_ret / subset["tb_return"].std() * np.sqrt(252)
    
    results.append({
        "year": test_year,
        "trades": len(subset),
        "mean_ret": mean_ret,
        "sharpe": sharpe,
        "max_dd": subset["dd"].min()
    })
    
    all_rows.append(subset)

In [30]:
wf_summary = pd.DataFrame(results)

wf_equity = pd.concat(all_rows).sort_values("tb_exit_time")

wf_equity["cum_pnl"] = wf_equity["tb_return"].cumsum()
wf_equity["peak"] = wf_equity["cum_pnl"].cummax()
wf_equity["drawdown"] = wf_equity["cum_pnl"] - wf_equity["peak"]

overall = (
    wf_equity["tb_return"].mean(),
    wf_equity["tb_return"].mean() / wf_equity["tb_return"].std() * np.sqrt(252)
        if wf_equity["tb_return"].std() > 0 else np.nan,
    wf_equity["drawdown"].min()
)

wf_summary, overall

(   year  trades  mean_ret    sharpe    max_dd
 0  2020      14 -0.002623 -0.776881 -0.180573
 1  2021       8  0.025585  4.741223 -0.076810
 2  2022       2 -0.019397 -6.568984  0.000000
 3  2023      17  0.002672  1.000752 -0.217485
 4  2024      15  0.034120  8.620344 -0.069199,
 (np.float64(0.012256672293737485),
  np.float64(3.3088029131323378),
  np.float64(-0.24622375206614125)))

In [31]:
# subset = test[
#     (test["strong_regime"] == 1) &
#     (test["risk_regime"] == 0) &   # <-- NEW
#     (test["prob"] >= THRESHOLD)
# ].copy()

In [32]:
# --- Parameters ---
HOLDING = 5
TP_MULT = 2.0   # profit (downward move)
SL_MULT = 1.0   # loss (upward move)

df = df.copy()

df["sigma"] = df["log_return"].rolling(20).std()

labels_short = [np.nan] * len(df)
exit_times_short = [pd.NaT] * len(df)
returns_short = [np.nan] * len(df)

close = df["close"].values
ts = df["ts"].values
sigma = df["sigma"].values

n = len(df)

for i in range(n - HOLDING):
    
    entry_px = close[i]
    entry_sigma = sigma[i]
    
    if np.isnan(entry_sigma):
        continue
    
    # --- NOTE inversion ---
    tp = entry_px * (1 - TP_MULT * entry_sigma)   # downward profit
    sl = entry_px * (1 + SL_MULT * entry_sigma)   # upward loss
    
    label = 0
    exit_time = ts[i + HOLDING]
    ret = 0
    
    for j in range(1, HOLDING + 1):
        px = close[i + j]
        
        # TP (price falls)
        if px <= tp:
            label = 1
            exit_time = ts[i + j]
            ret = (entry_px - px) / entry_px   # <-- IMPORTANT (short return)
            break
        
        # SL (price rises)
        elif px >= sl:
            label = 0
            exit_time = ts[i + j]
            ret = (entry_px - px) / entry_px
            break
        
        # timeout
        if j == HOLDING:
            px = close[i + j]
            ret = (entry_px - px) / entry_px
    
    labels_short[i] = label
    exit_times_short[i] = exit_time
    returns_short[i] = ret

# --- Assign ---
df["tb_label_short"] = labels_short
df["tb_exit_time_short"] = exit_times_short
df["tb_return_short"] = returns_short

df = df.dropna(subset=["tb_label_short"]).reset_index(drop=True)

In [33]:
df["tb_label_short"].value_counts(normalize=True)

tb_label_short
0.0    0.817521
1.0    0.182479
Name: proportion, dtype: float64

In [34]:
df[["tb_label", "tb_label_short"]].corr()

,tb_label,tb_label_short
tb_label,1.000000,-0.265673
tb_label_short,-0.265673,1.000000


In [35]:
train_df = df.copy()

features = [
    "skew_20",
    "vol_zscore",
    "ret_14",
    "ret_7",
    "trend_strength",
    "strong_regime"
]

train_df = train_df.dropna(subset=features + ["tb_label_short"]).reset_index(drop=True)

In [36]:
split_idx = int(len(train_df) * 0.8)

train = train_df.iloc[:split_idx]
test  = train_df.iloc[split_idx:]

X_train = train[features]
y_train = train["tb_label_short"]

X_test = test[features]
y_test = test["tb_label_short"]

In [37]:
from xgboost import XGBClassifier

model_short = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model_short.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [38]:
test = test.copy()
test["prob_short"] = model_short.predict_proba(X_test)[:, 1]

In [39]:
test.groupby("tb_label_short")[features].mean()

,skew_20,vol_zscore,ret_14,ret_7,trend_strength,strong_regime
tb_label_short,,,,,,
0.0,0.203746,-0.008896,0.036677,0.017981,0.021628,0.083527
1.0,0.120693,0.053445,0.041207,0.023033,0.030300,0.054348


In [40]:
test["prob_long"] = model.predict_proba(test[features])[:, 1]
test["prob_short"] = model_short.predict_proba(test[features])[:, 1]

In [41]:
test["edge"] = test["prob_long"] - test["prob_short"]

In [42]:
DELTA = 0.10
P_MIN = 0.30

def decide(row):
    pL = row["prob_long"]
    pS = row["prob_short"]
    
    edge = pL - pS
    
    if (edge > DELTA) and (pL > P_MIN):
        return 1
    elif (edge < -DELTA) and (pS > P_MIN):
        return -1
    else:
        return 0

test["signal"] = test.apply(decide, axis=1)

In [43]:
def get_ret(row):
    if row["signal"] == 1:
        return row["tb_return"]
    elif row["signal"] == -1:
        return row["tb_return_short"]
    else:
        return 0

test["ret"] = test.apply(get_ret, axis=1)

In [44]:
trades = test[test["signal"] != 0].copy()

trades = trades.sort_values("ts")

trades["cum"] = trades["ret"].cumsum()
trades["peak"] = trades["cum"].cummax()
trades["dd"] = trades["cum"] - trades["peak"]

mean_ret = trades["ret"].mean()

sharpe = (
    mean_ret / trades["ret"].std() * np.sqrt(252)
    if trades["ret"].std() > 0 else np.nan
)

max_dd = trades["dd"].min()

mean_ret, sharpe, max_dd, len(trades)

(np.float64(0.02634976342977895),
 np.float64(8.25770564138818),
 np.float64(-0.2946682541772452),
 195)

In [45]:
test["signal"].value_counts()

signal
 0    328
 1    141
-1     54
Name: count, dtype: int64

In [46]:
# PARAM_GRID = {
#     "delta": [0.12, 0.06, 0.08,0.1],
#     "max_depth": [2, 3],
#     "learning_rate": [0.03, 0.05]
# }

In [47]:
# from itertools import product

# grid = list(product(
#     PARAM_GRID["delta"],
#     PARAM_GRID["max_depth"],
#     PARAM_GRID["learning_rate"]
# ))

In [48]:
# best_score = -np.inf
# best_config = None
# MIN_TRADES = 10   # or 30 minimum


# for delta, depth, lr in grid:
#     y_train_long = train["tb_label"]
#     y_train_short = train["tb_label_short"]
#     # --- train models ---
#     model_long = XGBClassifier(
#         n_estimators=200,
#         max_depth=depth,
#         learning_rate=lr,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         random_state=42,
#         eval_metric="logloss"
#     )
    
#     model_short = XGBClassifier(
#         n_estimators=200,
#         max_depth=depth,
#         learning_rate=lr,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         random_state=42,
#         eval_metric="logloss"
#     )
    
#     model_long.fit(X_train, y_train_long)
#     model_short.fit(X_train, y_train_short)
    
#     # --- predict ---
#     prob_long = model_long.predict_proba(X_train)[:, 1]
#     prob_short = model_short.predict_proba(X_train)[:, 1]
    
#     edge = prob_long - prob_short
    
#     signal = np.where(edge > delta, 1,
#               np.where(edge < -delta, -1, 0))
    
#     ret = np.where(signal == 1, train["tb_return"],
#           np.where(signal == -1, train["tb_return_short"], 0))
    
#     if ret.std() == 0:
#         continue
#     # MIN_TRADES = 50   # or 30 minimum

#     if len(ret) < MIN_TRADES:
#         continue
#     sharpe = ret.mean() / ret.std() * np.sqrt(252)
    
#     score = sharpe * np.log(len(ret))
#     if score > best_score:
#         best_config = (delta,depth,lr)
    

In [49]:
delta, depth, lr = best_config
delta,depth,lr

NameError: name 'best_config' is not defined

In [50]:
trades = trades.sort_values("ts")

trades["cum_pnl"] = trades["ret"].cumsum()

mean_ret = trades["ret"].mean()

total_pnl = trades["ret"].sum()

win_rate = (trades["ret"] > 0).mean()

avg_win = trades[trades["ret"] > 0]["ret"].mean()
avg_loss = trades[trades["ret"] <= 0]["ret"].mean()

profit_factor = abs(avg_win / avg_loss) if avg_loss != 0 else np.nan


In [51]:
print({
    "trades": len(trades),
    "total_pnl": total_pnl,
    "mean_ret": mean_ret,
    "win_rate": win_rate,
    "avg_win": avg_win,
    "avg_loss": avg_loss,
    "profit_factor": profit_factor
})

{'trades': 195, 'total_pnl': np.float64(5.138203868806896), 'mean_ret': np.float64(0.02634976342977895), 'win_rate': np.float64(0.6512820512820513), 'avg_win': np.float64(0.057340098170578266), 'avg_loss': np.float64(-0.03152924410083152), 'profit_factor': np.float64(1.8186321875400124)}


In [52]:
trades.groupby("year")["ret"].agg(["count", "mean", "sum"])

,count,mean,sum
year,,,
2023,54,0.026051,1.406742
2024,137,0.028574,3.914702
2025,4,-0.045810,-0.183240


In [53]:
import joblib
joblib.dump(model, "model_long.pkl")
joblib.dump(model_short, "model_short.pkl")

['model_short.pkl']

In [54]:
import numpy as np
import pandas as pd
import joblib

# -----------------------
# CONFIG
# -----------------------
FEATURES = [
    "skew_20",
    "vol_zscore",
    "ret_14",
    "ret_7",
    "trend_strength",
    "strong_regime"
]

TP = 0.02
SL = 0.01
MAX_HOLD = 5

# -----------------------
# LOAD DATA
# -----------------------
df = pd.read_parquet("data/unseen_data.parquet")
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df = df.sort_values("ts").reset_index(drop=True)

# -----------------------
# FEATURE ENGINEERING
# -----------------------
df["log_return"] = np.log(df["close"] / df["close"].shift(1))

df["ret_7"] = df["close"].pct_change(7)
df["ret_14"] = df["close"].pct_change(14)

df["vol_20"] = df["log_return"].rolling(20).std()
df["vol_60"] = df["log_return"].rolling(60).std()
df["vol_zscore"] = (df["vol_20"] - df["vol_60"]) / df["vol_60"]

df["skew_20"] = df["log_return"].rolling(20).skew()

df["ma_10"] = df["close"].rolling(10).mean()
df["trend_strength"] = (df["close"] - df["ma_10"]) / df["ma_10"]

df["strong_regime"] = (df["trend_strength"] > 0).astype(int)

# -----------------------
# TRIPLE BARRIER
# -----------------------
df["tb_return"] = np.nan
df["tb_return_short"] = np.nan

close = df["close"].values

for i in range(len(df)):
    entry = close[i]
    
    tp_long = entry * (1 + TP)
    sl_long = entry * (1 - SL)
    
    tp_short = entry * (1 - TP)
    sl_short = entry * (1 + SL)
    
    end = min(i + MAX_HOLD, len(df) - 1)
    
    exit_long = entry
    exit_short = entry
    
    for j in range(i+1, end+1):
        price = close[j]
        
        if price >= tp_long or price <= sl_long:
            exit_long = price
            break
        
        if price <= tp_short or price >= sl_short:
            exit_short = price
            break
        
        exit_long = price
        exit_short = price
    
    df.loc[i, "tb_return"] = (exit_long - entry) / entry
    df.loc[i, "tb_return_short"] = (entry - exit_short) / entry

# -----------------------
# CLEAN
# -----------------------
df = df.dropna(subset=FEATURES + ["tb_return", "tb_return_short"])

# -----------------------
# LOAD MODELS
# -----------------------
model_long = joblib.load("model_long.pkl")
model_short = joblib.load("model_short.pkl")

# -----------------------
# PREDICT
# -----------------------
X = df[FEATURES]

df["prob_long"]  = model_long.predict_proba(X)[:, 1]
df["prob_short"] = model_short.predict_proba(X)[:, 1]

# -----------------------
# DSS (BEST CONFIG)
# -----------------------
edge = df["prob_long"] - df["prob_short"]

df["signal"] = 0

df.loc[
    (edge > 0.10) & (df["prob_long"] > 0.30),
    "signal"
] = 1

df.loc[
    (edge < -0.10) & (df["prob_short"] > 0.30),
    "signal"
] = -1

# -----------------------
# RETURNS
# -----------------------
df["ret"] = 0.0

df.loc[df["signal"] == 1, "ret"] = df["tb_return"]
df.loc[df["signal"] == -1, "ret"] = df["tb_return_short"]

trades = df[df["signal"] != 0].copy()

# -----------------------
# METRICS
# -----------------------
summary = {
    "trades": len(trades),
    "total_pnl": trades["ret"].sum(),
    "mean_ret": trades["ret"].mean(),
    "win_rate": (trades["ret"] > 0).mean(),
    "avg_win": trades[trades["ret"] > 0]["ret"].mean(),
    "avg_loss": trades[trades["ret"] <= 0]["ret"].mean(),
}

summary["profit_factor"] = (
    abs(summary["avg_win"] / summary["avg_loss"])
    if summary["avg_loss"] != 0 else np.nan
)

# -----------------------
# EQUITY
# -----------------------
trades = trades.sort_values("ts")

trades["cum"] = trades["ret"].cumsum()
trades["peak"] = trades["cum"].cummax()
trades["dd"] = trades["cum"] - trades["peak"]

print(summary)
print("Max Drawdown:", trades["dd"].min())

{'trades': 108, 'total_pnl': np.float64(-0.5008350127590537), 'mean_ret': np.float64(-0.0046373612292504975), 'win_rate': np.float64(0.26851851851851855), 'avg_win': np.float64(0.017002426332603134), 'avg_loss': np.float64(-0.012581080713981578), 'profit_factor': np.float64(1.3514281260200514)}
Max Drawdown: -0.583449927906013
